In [5]:
import pandas as pd
from pathlib import Path

files = sorted(Path("/Users/kellyg/eurogate-twin-1/eurogate_daily").glob("*.csv"))

for f in files[:5]:
    df = pd.read_csv(f, nrows=5, encoding="latin1")
    print(f.name)
    print(df.columns.tolist())
    print(df.head())
    print("-"*100)

Containerdaten-2024-10-27_01.55.36.csv
['containerId', 'customsRefId', 'lineCode', 'iedCode', 'sizetypeIsoCode', 'typeCode', 'reefer', 'teu', 'mgw', 'gross', 'csc', 'originOfTransportCode', 'arrivalTime', 'arrivalPolCode', 'arrivalType', 'arrivalLocationX', 'arrivalLocationY', 'arrivalLocationZ', 'arrivalVoyageEta', 'arrivalVesselName', 'arrivalServiceName', 'arrivalServiceCode', 'departureTime', 'departureType', 'destinationCode', 'released', 'forwarderName', 'numberOfArrivalContainersOnVessel', 'numberOfRemainOnBoardContainersOnVessel', 'numberOfDepartureContainersOnVessel']
   containerId           customsRefId  lineCode iedCode sizetypeIsoCode  \
0  TCNU2197105  ATB151468130920244851        77  IMPORT            45G1   
1  TGBU8605210  ATB150644481020244851        32  IMPORT            45G1   
2  GCXU6111154  ATB150553101020244851        77  IMPORT            45G1   
3  MEDU8987629  ATB150630421020244851        68  IMPORT            45G1   
4  CMAU5670653  ATB150564421020244851    

In [10]:
# # !pwd
# print(df.shape)
# print(df.columns.tolist())

import pandas as pd

df1 = pd.read_csv(files[0], encoding="latin1")
df2 = pd.read_csv(files[1], encoding="latin1")

overlap = set(df1["containerId"]).intersection(
    set(df2["containerId"])
)
# print(df1)
print(len(overlap))

17607


In [11]:
import pandas as pd
from pathlib import Path

folder = Path("/Users/kellyg/eurogate-twin-1/eurogate_daily")
files = sorted(folder.glob("*.csv"))

daily_dfs = []

for f in files:
    df = pd.read_csv(f, encoding="latin1")
    
    # Extract snapshot date from filename
    # Example: Containerdaten-2024-10-27_01.55.36.csv
    snapshot_str = f.name.replace("Containerdaten-", "").split("_")[0]
    df["snapshot_date"] = pd.to_datetime(snapshot_str)
    df["source_file"] = f.name
    
    daily_dfs.append(df)

all_snapshots = pd.concat(daily_dfs, ignore_index=True)

print(all_snapshots.shape)
print(all_snapshots["snapshot_date"].min(), all_snapshots["snapshot_date"].max())
print(all_snapshots["containerId"].nunique())

# Parse dates
for col in ["arrivalTime", "departureTime", "arrivalVoyageEta"]:
    all_snapshots[col] = pd.to_datetime(all_snapshots[col], errors="coerce")

# Sort by container and snapshot
all_snapshots = all_snapshots.sort_values(["containerId", "snapshot_date"])

container_history = (
    all_snapshots
    .groupby("containerId")
    .agg(
        first_seen=("snapshot_date", "min"),
        last_seen=("snapshot_date", "max"),
        n_snapshots=("snapshot_date", "nunique"),

        arrivalTime=("arrivalTime", "min"),
        departureTime=("departureTime", "max"),
        arrivalVoyageEta=("arrivalVoyageEta", "min"),

        sizetypeIsoCode=("sizetypeIsoCode", "first"),
        typeCode=("typeCode", "first"),
        reefer=("reefer", "first"),
        teu=("teu", "first"),
        gross=("gross", "first"),
        lineCode=("lineCode", "first"),
        arrivalType=("arrivalType", "first"),
        departureType=("departureType", "first"),
        originOfTransportCode=("originOfTransportCode", "first"),
        arrivalPolCode=("arrivalPolCode", "first"),
        destinationCode=("destinationCode", "first"),
        released=("released", "last"),

        n_unique_locations_x=("arrivalLocationX", "nunique"),
        n_unique_locations_y=("arrivalLocationY", "nunique"),
        n_unique_locations_z=("arrivalLocationZ", "nunique"),
    )
    .reset_index()
)

container_history["observed_days"] = (
    container_history["last_seen"] - container_history["first_seen"]
).dt.days + 1

container_history["dwell_hours"] = (
    container_history["departureTime"] - container_history["arrivalTime"]
).dt.total_seconds() / 3600

container_history["vessel_eta_delay_hours"] = (
    container_history["arrivalTime"] - container_history["arrivalVoyageEta"]
).dt.total_seconds() / 3600

container_history.head()

loc_cols = ["arrivalLocationX", "arrivalLocationY", "arrivalLocationZ"]

all_snapshots["location_key"] = (
    all_snapshots[loc_cols]
    .astype(str)
    .agg("_".join, axis=1)
)

movement_counts = (
    all_snapshots
    .sort_values(["containerId", "snapshot_date"])
    .assign(prev_location=lambda x: x.groupby("containerId")["location_key"].shift())
)

movement_counts["location_changed"] = (
    movement_counts["location_key"] != movement_counts["prev_location"]
) & movement_counts["prev_location"].notna()

rehandle_summary = (
    movement_counts
    .groupby("containerId")
    .agg(
        n_observed_location_changes=("location_changed", "sum"),
        n_unique_locations=("location_key", "nunique")
    )
    .reset_index()
)

container_history = container_history.merge(rehandle_summary, on="containerId", how="left")

container_history.to_csv("eurogate_container_history.csv", index=False)
all_snapshots.to_csv("eurogate_all_snapshots.csv", index=False)

(1138839, 32)
2024-10-27 00:00:00 2025-05-06 00:00:00
146674
